In [0]:
# confirm Spark session is live
print(f"Spark version: {spark.version}")
print("Running on serverless compute (SparkContext not available)")

In [0]:
# Eplore DBFS, Databricks' built-in distributed file system
display(dbutils.fs.ls("/"))

In [0]:
# Create EconMate volume and directory structure
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.econmate")

volume_base = "/Volumes/workspace/default/econmate"
dbutils.fs.mkdirs(f"{volume_base}/bronze/")
dbutils.fs.mkdirs(f"{volume_base}/silver/")
dbutils.fs.mkdirs(f"{volume_base}/gold/")

# Verify
display(dbutils.fs.ls(volume_base))

In [0]:
# Imports
# requests: for calling the World Bank HTTP API
# pyspark.sql.types: lets us define an explicit schema (better than inference)
# pyspark.sql.functions: col, lit, current_timestamp — your most-used PySpark tools

import requests
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType
)
from pyspark.sql.functions import col, lit, current_timestamp

print("Imports successful")

In [0]:
# We define which countries and what data we want

COUNTRIES = "KE;UG;TZ;ET;GH;NG;ZA;SN;ML;MZ;RW;ZM;ZW;CM;CI"
INDICATOR  = "NY.GDP.PCAP.CD"   # GDP per capita, current USD
BASE_URL   = "https://api.worldbank.org/v2/country"

print(f"Targeting {len(COUNTRIES.split(';'))} countries")
print(f"Indicator: {INDICATOR}")

In [0]:
# The API call function
# We wrap it in a function so it's reusable and testable

def fetch_world_bank(indicator: str, countries: str) -> list:
    """
    Fetches indicator data from the World Bank API.
    
    Returns a flat list of raw record dicts.
    World Bank response structure:
        response[0] = metadata (page info, total records)
        response[1] = the actual data records
    """
    url = f"{BASE_URL}/{countries}/indicator/{indicator}"
    
    params = {
        "format":   "json",
        "per_page": 500,       # pull up to 500 records in one call
        "date":     "2010:2023"  # 13 years of data
    }
    
    print(f"Calling: {url}")
    response = requests.get(url, params=params, timeout=30)
    
    # raise_for_status() throws an exception if HTTP 4xx or 5xx
    response.raise_for_status()
    
    data     = response.json()
    records  = data[1] if data and len(data) > 1 else []
    
    print(f"Raw records returned: {len(records)}")
    return records

# Call it
raw_records = fetch_world_bank(INDICATOR, COUNTRIES)

In [0]:
# Inspect one raw record so you understand the shape
import json
print(json.dumps(raw_records[0], indent=2))

In [0]:
# Flatten raw records into a clean list of dicts
# We extract only what we need and skip nulls (years with no data)

def flatten_records(records: list) -> list:
    """
    Flattens nested World Bank API response into
    a flat list of row-level dicts ready for a DataFrame.
    """
    flat = []
    for r in records:
        if r.get("value") is None:
            continue  # skip years where the indicator wasn't reported
        flat.append({
            "country_code":    r["country"]["id"],          # "KE"
            "country_name":    r["country"]["value"],        # "Kenya"
            "country_iso3":    r.get("countryiso3code", ""), # "KEN"
            "indicator_id":    r["indicator"]["id"],         # "NY.GDP.PCAP.CD"
            "indicator_name":  r["indicator"]["value"],
            "year":            int(r["date"]),
            "value":           float(r["value"]),
        })
    return flat

flat_data = flatten_records(raw_records)
print(f"Total records after flattening: {len(flat_data)}")
print(f"Sample: {flat_data[0]}")

In [0]:
# Define schema and create a Spark DataFrame

schema = StructType([
    StructField("country_code",   StringType(),  True),
    StructField("country_name",   StringType(),  True),
    StructField("country_iso3",   StringType(),  True),
    StructField("indicator_id",   StringType(),  True),
    StructField("indicator_name", StringType(),  True),
    StructField("year",           IntegerType(), True),
    StructField("value",          DoubleType(),  True),
])

df = spark.createDataFrame(flat_data, schema=schema)

# Add ingestion metadata columns
# current_timestamp() = when this record entered our pipeline
# lit() = attach a literal string as a column
df = (df
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source",      lit("world_bank_api"))
    .withColumn("indicator",   lit("gdp_per_capita"))
)

print(f"DataFrame created with {df.count()} rows and {len(df.columns)} columns")
df.printSchema()

In [0]:
# Write to Bronze as a Unity Catalog managed table

(df.write
   .format("delta")
   .mode("overwrite")
   .option("mergeSchema", "true")
   .saveAsTable("workspace.default.bronze_world_bank_gdp")
)

print("Written to Unity Catalog table: workspace.default.bronze_world_bank_gdp")

In [0]:
# Verify
df_check = spark.table("workspace.default.bronze_world_bank_gdp")
print(f"Total rows: {df_check.count()}")
display(df_check.groupBy("country_code", "country_name").count().orderBy("country_code"))